# Analysis & Analytics

## Data Acquisition

### Prepartion

#### Imports

In [1]:
from nyc_taxi_routes.notebook import *
setup_plotting()

✓  wgnd theme activated (matplotlib · seaborn)

#### Constants

In [2]:
DATA_PATH = PATHS["processed"]
DATA_FILE= 'ny-taxi-routes_prep.parquet'

### Data Gathering

In [5]:
df_prep= df = pd.read_parquet(DATA_PATH / DATA_FILE, engine='pyarrow')

df_final= df_prep.copy()

df_final.head()

,pickup_weekday,pickup_hour,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,trip_distance,fare_amount,tip_amount,tolls_amount,payment_type,departure,arrival,route,total_yield,price_per_mile,has_tolls,time_slot,is_weekend,trip_distance_log,fare_amount_log,total_yield_log,price_per_mile_log
0,3,19,-73.789970,40.646660,-74.005051,40.748081,1,18.610001,52.0,10.00,5.54,1,JFK,NYC,JFK-NYC,62.00,3.331542,1,Evening Rush,0,2.976040,3.970292,4.143135,1.465924
1,5,3,-73.986237,40.746513,-73.996796,40.742504,1,0.990000,5.0,1.00,0.00,1,NYC,NYC,NYC-NYC,6.00,6.060606,0,Night,1,0.688135,1.791759,1.945910,1.954531
2,4,20,-73.874634,40.773960,-73.959923,40.762802,3,9.250000,26.5,8.34,5.54,1,NYC,NYC,NYC-NYC,34.84,3.766486,1,Evening Rush,0,2.327278,3.314186,3.579065,1.561609
3,5,2,-73.952477,40.772064,-73.949371,40.675156,1,9.200000,28.0,0.00,0.00,2,NYC,NYC,NYC-NYC,28.00,3.043478,0,Night,1,2.322388,3.367296,3.367296,1.397105
4,4,21,-73.988281,40.764488,-73.996513,40.753239,1,0.900000,5.0,1.26,0.00,1,NYC,NYC,NYC-NYC,6.26,6.955556,0,Late Night,0,0.641854,1.791759,1.982380,2.073871


## Business Questions

**1)** [Anteil JFK Departures](###-Anteil-JFK-Departures)   
Wie hoch ist der Anteil an Taxis, die vom Flughafen (JFK) aus gebucht werden insgesamt? 

**2)** Wo werden Taxis in New York genommen? Erstelle eine Visualisierung der Startpunkte der Taxifahrten.  

**3)** Wie hoch ist der Anteil an Taxis, die vom Flughafen aus gebucht werden pro Wochentag? An welchem Wochentag gibt es den höchsten Anteil und wann den niedrigsten?

**4)** Erstelle eine Visualisierung, anhand derer man sieht, welchen Anteil ein **Wochentag** an der Anzahl der Fahrten insgesamt hat. Dies soll sowohl für die Fahrten vom Flughafen aus gemacht werden als auch für das Gesamtset.  

**5)** Erstelle eine Visualisierung, anhand derer man sieht, welchen Anteil eine **Uhrzeit** an der Anzahl der Fahrten insgesamt hat. Dies soll sowohl für die Fahrten vom Flughafen aus gemacht werden als auch für das Gesamtset.  

### Anteil JFK Departures

In [ ]:
# NEW DATA FRAME FOR ROUTES WITH GROUP BY

inspect(df_final,['dimensions'])

section_header('Routes Overview')
df_routes = df_final.groupby(["route"], observed=False ).agg(
    trip_sum=("trip_distance", "count"),
    dist_sum=("trip_distance", "sum") ,
    fare_sum=("fare_amount", "sum")
)
df_routes['trip_pct'] = (df_routes['trip_sum'] / df_routes['trip_sum'].sum()) * 100
df_routes['fare_pct'] = (df_routes['fare_sum'] / df_routes['fare_sum'].sum()) * 100
df_routes['dist_pct'] = (df_routes['dist_sum'] / df_routes['dist_sum'].sum()) * 100
show_df(df_routes)

section_header('JFK Depatures')
df_jfk_dep = df_final.groupby("departure", observed=False).size().reset_index(name='cnt')
df_jfk_dep['pct'] = (df_jfk_dep['cnt'] / df_jfk_dep['cnt'].sum()) * 100
show_df(df_jfk_dep)

total_trips = df_final.shape[0]
df_dep_arr_jfk_sum = df_routes.loc[['JFK-JFK', 'JFK-NYC', 'JFK-OTHER', 'NYC-JFK'], :]["trip_sum"].sum()
dep_arr_jfk_pct = round( (df_dep_arr_jfk_sum * 100) /  total_trips,2)


success(f'Anteil der Abfahrten am JFK beträgt {df_jfk_dep.loc[0, "pct"]/100:.2%}')
log(f'\n- Gesamtanzahl aller Fahrten: {total_trips}')
log(f"- Gesamtnzahl der Fahrten zum und vom Flughafen: {df_dep_arr_jfk_sum}")
log(f"- Gesamtanteil der Fahrten zum und vom Flughafen:  {dep_arr_jfk_pct}%")




───  DIMENSIONS  ─────────────────────────────────────────────


,metric,count,pct
0,rows,293369,
1,columns,24,
2,duplicates,0,0.0%
3,empty rows (all NaN),0,0.0%
4,empty cols (all NaN),0,0.0%



───  ROUTES OVERVIEW  ────────────────────────────────────────


,trip_sum,dist_sum,fare_sum,trip_pct,fare_pct,dist_pct
route,,,,,,
JFK-JFK,143,577.060,2951.060,0.05%,0.08%,0.07%
JFK-NYC,5445,89246.930,253069.500,1.86%,7.01%,10.65%
JFK-OTHER,61,1198.000,3928.000,0.02%,0.11%,0.14%
NYC-JFK,1964,34670.840,100673.500,0.67%,2.79%,4.14%
NYC-NYC,285174,702059.250,3207394.250,97.21%,88.90%,83.77%
NYC-OTHER,551,10092.550,38407.949,0.19%,1.06%,1.20%
OTHER-NYC,1,25.200,73.000,0.00%,0.00%,0.00%
OTHER-OTHER,30,170.470,1194.750,0.01%,0.03%,0.02%



───  JFK DEPATURES  ──────────────────────────────────────────


,departure,cnt,pct
0,JFK,5649,1.93%
1,NYC,287689,98.06%
2,OTHER,31,0.01%


✓  Anteil der Abfahrten am JFK beträgt 1.93%

- Gesamtanzahl aller Fahrten: 293369
- Gesamtnzahl der Fahrten zum und vom Flughafen: 7613
- Gesamtanteil der Fahrten zum und vom Flughafen:  2.6 %


## Open Next Steps

Aus der Aufgabenstellung (`docs/infos.md`) noch offen:

- Visualisierung der Pickup-Standorte in NYC (Frage 4)
- JFK-Anteil pro Wochentag + Visualisierung (Frage 5)
- Wochentags-Verteilung Gesamt vs. JFK (Frage 6)
- Uhrzeit-Verteilung Gesamt vs. JFK (Frage 7)
- Empfehlung an den Taxiunternehmer formulieren (Frage 9)
